<a href="https://colab.research.google.com/github/AlexitoFernandez/practicas-google-colab/blob/unidad-3/Practica_5_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#"Machine Learning"
##Unidad III
### **Practica 5 - Random Forest Prestamosa Lending Club**

Alumno: Jorge Alejandro Fernández De Los Santos.

Facilitador: José Gabriel Rodríguez Rivas.


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
import gdown

# Cargar datos
file_id = '1HBU_efbbPg378pZvLZ1AtvRFAKrEFsob'
output = 'lending_club_2007_2011_6_states.csv'
gdown.download(id=file_id, output=output, quiet=False)
prestamos_df= pd.read_csv(output)
print(prestamos_df.head())

Downloading...
From: https://drive.google.com/uc?id=1HBU_efbbPg378pZvLZ1AtvRFAKrEFsob
To: /content/lending_club_2007_2011_6_states.csv
100%|██████████| 7.01M/7.01M [00:00<00:00, 95.3MB/s]

   loan_amnt  funded_amnt  funded_amnt_inv       term  int_rate  installment  \
0       2400         2400           2400.0  36 months     15.96        84.33   
1      10000        10000          10000.0  36 months     13.49       339.31   
2       3000         3000           3000.0  36 months     18.64       109.43   
3       5600         5600           5600.0  60 months     21.28       152.39   
4       5375         5375           5350.0  60 months     12.69       121.45   

  grade sub_grade            emp_title emp_length  ... application_type  \
0     C        C5                  NaN  10+ years  ...       Individual   
1     C        C1  AIR RESOURCES BOARD  10+ years  ...       Individual   
2     E        E1      MKC Accounting     9 years  ...       Individual   
3     F        F2                  NaN    4 years  ...       Individual   
4     B        B5            Starbucks   < 1 year  ...       Individual   

   acc_now_delinq chargeoff_within_12_mths delinq_amnt pub_rec_bankr

In [2]:
#Transformación de datos
if 'grade' in prestamos_df.columns: prestamos_df['grade_code'] = prestamos_df['grade'].factorize()[0]
if 'purpose' in prestamos_df.columns: prestamos_df['purpose_code'] = prestamos_df['purpose'].factorize()[0]
if 'addr_state' in prestamos_df.columns: prestamos_df['addr_state_code'] = prestamos_df['addr_state'].factorize()[0]
if 'home_ownership' in prestamos_df.columns: prestamos_df['home_ownership_code'] = prestamos_df['home_ownership'].factorize()[0]
print("Datos transformados.")

#Los meses al estar en stack, se transforman de texto a número y después se dividen en 12 para transformarlos a años
prestamos_df['loan_term_year'] = prestamos_df['term'].str.extract(r'(\d+)').astype(int) / 12

#La Regresión Logística es un clasificador binario, por lo que necesita convertir estas categorías en algo que pueda medir
prestamos_df['repaid'] = prestamos_df['loan_status'].apply(lambda x: 1 if x == 'Fully Paid' else 0)

# Selección de variables predictoras
X = prestamos_df[['funded_amnt', 'loan_term_year', 'int_rate', 'grade_code',
                  'purpose_code', 'addr_state_code', 'home_ownership_code',
                  'annual_inc', 'dti', 'revol_util', 'pub_rec_bankruptcies']]

# Variable objetivo o variable a predecir
y = prestamos_df["repaid"]

Datos transformados.


In [4]:
# Dividimos el dataFrame
# stratify mantiene la misma proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size= 0.4, stratify=y )

In [5]:
# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11944, 11), (7964, 11), (11944,), (7964,))

In [6]:
# Creación del Regressor con parametros por defecto
rfc1 = RandomForestClassifier(random_state=23)
# Fit estimator and display score
rfc1 = rfc1.fit(X_train, y_train)
# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", rfc1.score(X_train, y_train) )

Precision del clasificador en fase de entrenamiento 1.0


In [7]:
# Realizar una prediccion con los datos de prueba
y_pred = rfc1.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte del clasificador Random Forest sin balanceo de clases : \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print(f'\nMatriz de Confusion Random Forest sin balanceo de clases:\n', confusion_matrix(y_test, y_pred ))


Reporte del clasificador Random Forest sin balanceo de clases : 
               precision    recall  f1-score   support

   No Pagado       0.32      0.03      0.06      1177
      Pagado       0.85      0.99      0.92      6787

    accuracy                           0.85      7964
   macro avg       0.59      0.51      0.49      7964
weighted avg       0.78      0.85      0.79      7964


Matriz de Confusion Random Forest sin balanceo de clases:
 [[  36 1141]
 [  75 6712]]


In [8]:
#Random Forest con balanceo de clases
rfc2 = RandomForestClassifier(class_weight='balanced', random_state=23)
rfc2 = rfc2.fit(X_train, y_train)
y_pred = rfc2.predict(X_test)
print("Reporte del clasificador con balanceo de clases: \n", classification_report(y_test, y_pred,
target_names=["No Pagado", "Pagado"] ))
print('Matriz de Confusion con balanceo de clases \n' , confusion_matrix(y_test, y_pred))

Reporte del clasificador con balanceo de clases: 
               precision    recall  f1-score   support

   No Pagado       0.38      0.02      0.03      1177
      Pagado       0.85      1.00      0.92      6787

    accuracy                           0.85      7964
   macro avg       0.62      0.51      0.48      7964
weighted avg       0.78      0.85      0.79      7964

Matriz de Confusion con balanceo de clases 
 [[  20 1157]
 [  32 6755]]


In [9]:
#Modelo 3 con n_estimators=200 y max_depth=7
rfc3 = RandomForestClassifier(n_estimators=200, max_depth=7, class_weight='balanced', random_state=23)
# Fit estimator and display score
rfc3 = rfc3.fit(X_train, y_train)
y_pred = rfc3.predict(X_test)
print("Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=7: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print('Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=7\n' ,
confusion_matrix(y_test, y_pred))

Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=7: 
               precision    recall  f1-score   support

   No Pagado       0.24      0.56      0.34      1177
      Pagado       0.90      0.70      0.79      6787

    accuracy                           0.68      7964
   macro avg       0.57      0.63      0.56      7964
weighted avg       0.80      0.68      0.72      7964

Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=7
 [[ 665  512]
 [2065 4722]]


In [11]:
#Modelo 4 con n_estimators=200 y max_depth=10
rfc4 = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=23)
# Fit estimator and display score
rfc4 = rfc4.fit(X_train, y_train)
y_pred = rfc4.predict(X_test)
print("Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=10: \n",
classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print('Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=10\n' ,
confusion_matrix(y_test, y_pred))

Reporte del clasificador con balanceo de clases n_estimators=200, max_depth=10: 
               precision    recall  f1-score   support

   No Pagado       0.26      0.35      0.30      1177
      Pagado       0.88      0.83      0.85      6787

    accuracy                           0.76      7964
   macro avg       0.57      0.59      0.58      7964
weighted avg       0.79      0.76      0.77      7964

Matriz de Confusion con balanceo de clases n_estimators=200, max_depth=10
 [[ 408  769]
 [1160 5627]]


In [13]:
import plotly.graph_objects as go

# --- Nombres de los modelos --
modelos = [
"RF sin balanceo",
"RF balanceado",
"RF bal. n=200, depth=7",
"RF bal. n=200, depth=10"
]

# --- Métricas reales --
accuracy = [0.85, 0.85, 0.65, 0.73]
recall_no_pagado = [0.03, 0.01, 0.58, 0.40]
recall_pagado = [0.99, 1.00, 0.66, 0.78]
f1_macro = [0.49, 0.47, 0.54, 0.57]

# --- Ejes --
metricas = ["Accuracy", "Recall No Pagado", "Recall Pagado", "F1 Macro"]

# --- Crear la figura --
fig = go.Figure()

# Agregar cada modelo al radar
for i, modelo in enumerate(modelos):
    fig.add_trace(go.Scatterpolar(
        r=[accuracy[i], recall_no_pagado[i], recall_pagado[i], f1_macro[i], accuracy[i]],
        theta=metricas + [metricas[0]],
        fill='toself',
        name=modelo
    ))

# --- Personalización --
fig.update_layout(
    title="Comparación de modelos Random Forest (Radar Plot)",
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1], tickvals=[0.2, 0.4, 0.6, 0.8, 1.0]),
    ),
    showlegend=True
)
fig.update_layout(width=800, height=600)
fig.show()

In [14]:
# Búsqueda de hiperparámetros con GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
# Iniciamos con un Modelo base con balanceo de clases (para compensar desbalance)
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
# --- Definimos el espacio de búsqueda --
param_grid = {
'n_estimators': [100, 200, 300],
'max_depth': [5, 7, 10 ],
'min_samples_split': [2, 5, 10],
'min_samples_leaf': [1, 2, 4],
'max_features': ['sqrt', 'log2']
}
# --- Configuración de GridSearchCV --
grid_search = GridSearchCV(
estimator=rf,
param_grid=param_grid,
scoring='f1_macro',
# Se puede cambiar a 'recall_macro', 'accuracy', etc.
cv=3,
n_jobs=-1,
verbose=2
)
# 3 particiones para validación cruzada
# usa todos los núcleos disponibles
# para ver el progreso
# --- Ejecutar búsqueda --
grid_search.fit(X_train, y_train)
# --- Mostrar los mejores parámetros --
print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)
print("\n Mejor puntaje promedio (validación cruzada):")
print(grid_search.best_score_)
# --- Entrenar modelo final con los mejores parámetros --
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
# --- Evaluar en conjunto de prueba --
print("\n Reporte de Clasificación (modelo óptimo):")
print(classification_report(y_test, y_pred_best, target_names=["No Pagado", "Pagado"]))
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred_best))


Fitting 3 folds for each of 162 candidates, totalling 486 fits
Mejores hiperparámetros encontrados:
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 200}

 Mejor puntaje promedio (validación cruzada):
0.5782153340467877

 Reporte de Clasificación (modelo óptimo):
              precision    recall  f1-score   support

   No Pagado       0.26      0.40      0.32      1177
      Pagado       0.89      0.81      0.84      6787

    accuracy                           0.75      7964
   macro avg       0.57      0.60      0.58      7964
weighted avg       0.79      0.75      0.77      7964

Matriz de Confusión:
[[ 469  708]
 [1312 5475]]


In [16]:
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

#GridSearchCV prioriza la clase minoritaria de la siguiente forma
scorer = make_scorer(recall_score, pos_label=0)
grid_search = GridSearchCV(
RandomForestClassifier(class_weight='balanced', random_state=23),
param_grid=param_grid,
scoring=scorer,  # << prioriza recall de la clase No Pagado
cv=5,
n_jobs=-1,
verbose=2
)

In [17]:
#Usando Usar un “scoring” balanceado: F2-score o weighted F1 para GridSearchCV priorizando la clase minotaria
from sklearn.metrics import make_scorer, fbeta_score
# F2 da más peso al recall (importante para detectar impagos)
scorer_f2 = make_scorer(fbeta_score, beta=2, pos_label=0)
grid_search = GridSearchCV(
RandomForestClassifier(class_weight='balanced', random_state=23),
param_grid=param_grid,
scoring=scorer_f2,
cv=5,
n_jobs=-1,
verbose=2
)

In [19]:
from sklearn.metrics import make_scorer, recall_score, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Usar múltiples métricas y evaluar todas
grid_search = GridSearchCV(
RandomForestClassifier(class_weight='balanced', random_state=23),
param_grid=param_grid,
scoring={
'recall_no_pagado': make_scorer(recall_score, pos_label=0),
'f1_no_pagado': make_scorer(f1_score, pos_label=0),
'accuracy': 'accuracy'
},
refit='recall_no_pagado',  # <--- elige el modelo que maximice recall de No Pagado
cv=5,
n_jobs=-1,
verbose=2
)

In [20]:
#Busqueda de parámetros optimos priorizando detección de NO pagados
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score, classification_report, confusion_matrix
# Definición del modelo base con balanceo de clases
# El parámetro class_weight='balanced' ajusta el peso inversamente proporcional
# a la frecuencia de cada clase, ayudando a tratar el desbalance.
modelo_base = RandomForestClassifier(
class_weight='balanced',
random_state=23
)
# Definición de la cuadrícula de hiperparámetros
param_grid = {
'n_estimators': [100, 200, 300],
'max_depth': [5, 7, 10],
'min_samples_split': [2, 5, 10],
'min_samples_leaf': [1, 2, 4],
'max_features': ['sqrt', 'log2']
}
# Definición del criterio de evaluación
# Se usará el recall de la clase "No Pagado" (pos_label=0)
# Esto hace que el modelo busque detectar el mayor número posible de impagos.
scorer = make_scorer(recall_score, pos_label=0)
# Configuración del GridSearchCV
grid_search = GridSearchCV(
estimator=modelo_base,
param_grid=param_grid,
scoring=scorer,
# prioriza el recall de la clase No Pagado
cv=5,
n_jobs=-1,
verbose=2
)
# validación cruzada con 5 particiones
# usa todos los núcleos disponibles
# Entrenamiento del modelo con búsqueda de hiperparámetros
grid_search.fit(X_train, y_train)
print("\nBúsqueda finalizada.")
print(f"Mejores hiperparámetros encontrados:\n{grid_search.best_params_}")
# Evaluación del mejor modelo
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nReporte de Clasificación (modelo optimizado para detectar impagos):\n")
print(classification_report(y_test, y_pred))
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

Fitting 5 folds for each of 162 candidates, totalling 810 fits

Búsqueda finalizada.
Mejores hiperparámetros encontrados:
{'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 300}

Reporte de Clasificación (modelo optimizado para detectar impagos):

              precision    recall  f1-score   support

           0       0.23      0.62      0.33      1177
           1       0.91      0.63      0.74      6787

    accuracy                           0.63      7964
   macro avg       0.57      0.63      0.54      7964
weighted avg       0.80      0.63      0.68      7964

Matriz de Confusión:
[[ 731  446]
 [2517 4270]]
